# BatchTopK k-sweep: FP8 vs FP16 vs SAEBench authors

Sparsity (k) sweep at a single width (2^16 = 65,536), for **both** SAEBench models
(`gemma-2-2b`, `pythia-160m-deduped`). For each model we plot the **frontier** — a
quality metric vs the achieved L0 (average active latents) — for three series:

* **FP8 (ours)** — our `batchtopk_fp8` runs (`results/saebench_<model>_fp8`)
* **FP16 (ours)** — our standard BatchTopK runs (`results/saebench_<model>`)
* **authors** — the published SAEBench BatchTopK baseline
  (`adamkarvonen/sae_bench_results_0125`, trainers 0–5) for the same model/width

Our k grid is `{50, 100, 250, 500, 1000, 2500}`; the authors sweep their own k
(trainers 0–5). On an L0 axis both trace the same frontier, so the comparison is
"do our FP8/FP16 points land on the authors' curve?". Train + eval with
`experiments/sweep_k.sh`. Missing runs are skipped, so this renders incrementally.

In [1]:
from pathlib import Path
import glob, json, re

import pandas as pd
import plotly.graph_objects as go
from huggingface_hub import hf_hub_download

pd.set_option("display.float_format", lambda v: f"{v:.4f}")


def dig(d, *path, default=None):
    """Safely walk nested dicts: dig(metrics, 'sparsity', 'l0')."""
    cur = d
    for p in path:
        if not isinstance(cur, dict) or p not in cur:
            return default
        cur = cur[p]
    return cur


# friendly_name -> (group, key, higher_is_better) within eval_result_metrics.
METRIC_REGISTRY = {
    "core": {
        "L0":            ("sparsity", "l0", None),
        "CE_loss_score": ("model_performance_preservation", "ce_loss_score", True),
        "KL_div_score":  ("model_behavior_preservation", "kl_div_score", True),
        "explained_var": ("reconstruction_quality", "explained_variance", True),
        "frac_alive":    ("misc_metrics", "frac_alive", True),
    },
    "sparse_probing": {
        "probe_top1_acc": ("sae", "sae_top_1_test_accuracy", True),
        "probe_top5_acc": ("sae", "sae_top_5_test_accuracy", True),
        "probe_full_acc": ("sae", "sae_test_accuracy", True),
        "llm_top1_acc":   ("llm", "llm_top_1_test_accuracy", True),
    },
}
EVAL_TYPES = ["core", "sparse_probing"]


def extract_metrics(eval_type, result_json):
    m = result_json.get("eval_result_metrics", {})
    return {name: dig(m, grp, key)
            for name, (grp, key, _) in METRIC_REGISTRY.get(eval_type, {}).items()}

In [2]:
# --- Config -----------------------------------------------------------------
RESULTS_ROOT = Path("/wekafs/smerrill/efficient_sae/experiments/results")
AUTHOR_REPO = "adamkarvonen/sae_bench_results_0125"

WIDTH = 65536
WIDTH_POW = 16
KS = [50, 100, 250, 500, 1000, 2500]
member_of = lambda k: f"w{WIDTH}_k{k}"

# short run-dir key (matches train_saebench_replication.py --model) ->
#   full model name used in the author result files + its resid layer + author token.
MODELS = {
    "gemma":  dict(full="gemma-2-2b",          layer=12, token="gemma-2-2b__0108"),
    "pythia": dict(full="pythia-160m-deduped", layer=8,  token="pythia-160m-deduped__0108"),
}
# series label -> run-dir suffix produced by the trainer.
OUR_PRECISIONS = {"FP16 (ours)": "", "FP8 (ours)": "_fp8"}
AUTHOR_TRAINERS = [0, 1, 2, 3, 4, 5]

print("models :", list(MODELS))
print("width  :", WIDTH, f"(2^{WIDTH_POW})")
print("k grid :", KS)

models : ['gemma', 'pythia']
width  : 65536 (2^16)
k grid : [50, 100, 250, 500, 1000, 2500]


In [3]:
# --- Load OUR results + the AUTHORS' baseline into one long dataframe --------
member_step_re = re.compile(r"_step_(\d+)_")


def load_our_member(run_dir: Path, member: str):
    """Long rows (eval_type, metric, step, value) for one trained member.

    Uses a precise '_<member>_step_' match so w65536_k50 doesn't also capture
    w65536_k500 (k50 vs k500, k100 vs k1000)."""
    rows = []
    eval_dir = run_dir / "saebench_eval" / member / "eval_results"
    if not eval_dir.exists():
        return rows
    for path in glob.glob(str(eval_dir / "*" / "*_eval_results.json")):
        p = Path(path)
        et = p.parent.name
        if et not in EVAL_TYPES or f"_{member}_step_" not in p.name:
            continue
        sm = member_step_re.search(p.name)
        step = int(sm.group(1)) if sm else -1
        data = json.loads(p.read_text())
        for metric, value in extract_metrics(et, data).items():
            rows.append(dict(eval_type=et, metric=metric, step=step, value=value))
    return rows


long_rows = []
for short, spec in MODELS.items():
    for series, sfx in OUR_PRECISIONS.items():
        run_dir = RESULTS_ROOT / f"saebench_{short}{sfx}"
        for k in KS:
            for row in load_our_member(run_dir, member_of(k)):
                long_rows.append(dict(model=short, series=series, point=f"k{k}", **row))

author_missing = []
for short, spec in MODELS.items():
    release = f"saebench_{spec['full']}_width-2pow{WIDTH_POW}_date-0108"
    for t in AUTHOR_TRAINERS:
        for et in EVAL_TYPES:
            fname = (f"{et}/{release}/{release}_BatchTopK_{spec['token']}"
                     f"_resid_post_layer_{spec['layer']}_trainer_{t}_eval_results.json")
            try:
                local = hf_hub_download(AUTHOR_REPO, fname, repo_type="dataset")
                data = json.loads(Path(local).read_text())
            except Exception as e:  # noqa: BLE001
                author_missing.append((short, t, et, str(e).splitlines()[0][:50]))
                continue
            for metric, value in extract_metrics(et, data).items():
                long_rows.append(dict(model=short, series="authors",
                                      point=f"trainer_{t}", eval_type=et,
                                      metric=metric, step=-1, value=value))

raw_df = pd.DataFrame(long_rows)
assert not raw_df.empty, (
    "No results found at all. Train+eval first, e.g.\n"
    "  cd experiments && ./sweep_k.sh        (or PHASE=eval CHECKPOINTS=final ./sweep_k.sh)"
)
if author_missing:
    print(f"authors missing ({len(author_missing)}):", author_missing[:6],
          "..." if len(author_missing) > 6 else "")

# Frontier = the FINAL checkpoint of each of our members (authors are single-point).
ours = raw_df[raw_df.step >= 0]
auth = raw_df[raw_df.step < 0]
ours_final = ours[
    ours.groupby(["model", "series", "point"])["step"].transform("max") == ours.step
]
frontier_long = pd.concat([ours_final, auth], ignore_index=True)
wide = (frontier_long
        .pivot_table(index=["model", "series", "point"], columns="metric", values="value")
        .reset_index())
print("our series present:", sorted(set(zip(ours.model, ours.series))))
wide

(…)ost_layer_12_trainer_0_eval_results.json: 0.00B [00:00, ?B/s]

(…)ost_layer_12_trainer_0_eval_results.json: 0.00B [00:00, ?B/s]

(…)ost_layer_12_trainer_1_eval_results.json: 0.00B [00:00, ?B/s]

(…)ost_layer_12_trainer_1_eval_results.json: 0.00B [00:00, ?B/s]

(…)ost_layer_12_trainer_3_eval_results.json: 0.00B [00:00, ?B/s]

...

(…)post_layer_8_trainer_4_eval_results.json: 0.00B [00:00, ?B/s]

(…)post_layer_8_trainer_5_eval_results.json: 0.00B [00:00, ?B/s]

(…)post_layer_8_trainer_5_eval_results.json: 0.00B [00:00, ?B/s]

our series present: []


metric,model,series,point,CE_loss_score,KL_div_score,L0,explained_var,frac_alive,llm_top1_acc,probe_full_acc,probe_top1_acc,probe_top5_acc
0,gemma,authors,trainer_0,0.9786,0.9778,21.1057,0.6680,0.8756,0.6535,0.9511,0.7398,0.8485
1,gemma,authors,trainer_1,0.9868,0.9862,41.9135,0.7227,0.8829,0.6535,0.9545,0.7436,0.8628
2,gemma,authors,trainer_2,0.9918,0.9912,84.3313,0.7734,0.8703,0.6535,0.9556,0.7393,0.8569
3,gemma,authors,trainer_3,0.9951,0.9943,168.6492,0.8164,0.8560,0.6535,0.9563,0.7728,0.8583
4,gemma,authors,trainer_4,0.9967,0.9964,339.3978,0.8594,0.7704,0.6535,0.9575,0.7124,0.8664
5,gemma,authors,trainer_5,0.9984,0.9979,695.5989,0.9023,0.6013,0.6535,0.9556,0.7776,0.8549
6,pythia,authors,trainer_0,0.9630,-1.0000,19.5486,0.8015,0.8152,0.6460,0.9367,0.7319,0.8218
7,pythia,authors,trainer_1,0.9763,-1.0000,38.6494,0.8434,0.6152,0.6460,0.9376,0.7459,0.8504
8,pythia,authors,trainer_2,0.9853,-1.0000,77.7574,0.8809,0.5495,0.6460,0.9391,0.7418,0.8528
9,pythia,authors,trainer_3,0.9917,-1.0000,157.9800,0.9184,0.4487,0.6460,0.9394,0.7722,0.8491


## Frontier plots — quality vs L0, per model

Each panel: FP8 (ours), FP16 (ours), authors. If our points sit on the authors'
curve, our recipe matches the field; if FP8 overlaps FP16, the 8-bit GEMMs cost
nothing in quality. L0 is on a log axis (k spans 50→2500).

In [4]:
COLORS = {"authors": "#7f7f7f", "FP16 (ours)": "#1f77b4", "FP8 (ours)": "#d62728"}
SERIES_ORDER = ["authors", "FP16 (ours)", "FP8 (ours)"]
Y_METRICS = [
    ("CE_loss_score", "CE loss recovered (higher better)"),
    ("explained_var", "explained variance (higher better)"),
    ("probe_top1_acc", "sparse-probing top-1 acc (higher better)"),
]

for short, spec in MODELS.items():
    md = wide[wide["model"] == short]
    if md.empty:
        print(f"[{short}] no data yet — train/eval it.")
        continue
    for ycol, ylabel in Y_METRICS:
        if ycol not in md.columns or md[ycol].isna().all():
            continue
        fig = go.Figure()
        for series in SERIES_ORDER:
            s = (md[md["series"] == series]
                 .dropna(subset=["L0", ycol]).sort_values("L0"))
            if s.empty:
                continue
            fig.add_trace(go.Scatter(
                x=s["L0"], y=s[ycol], mode="lines+markers", name=series,
                text=s["point"], hovertemplate="%{text}<br>L0=%{x:.1f}<br>"
                + ycol + "=%{y:.4f}<extra>" + series + "</extra>",
                marker=dict(size=9), line=dict(color=COLORS.get(series),
                            dash="dash" if series == "FP8 (ours)" else "solid")))
        if not fig.data:
            continue
        fig.update_layout(
            title=f"{spec['full']} — {ylabel} vs L0  (BatchTopK, width {WIDTH})",
            xaxis_title="L0 (avg active latents)", yaxis_title=ycol,
            xaxis_type="log", width=780, height=480,
            legend=dict(x=0.99, y=0.01, xanchor="right", yanchor="bottom"))
        fig.show()

## Checkpoint evolution (ours) + FP8-vs-FP16 frontier table

Below: each metric vs training step across the 5 checkpoints (one line per k,
FP8 dashed / FP16 solid), then a table of the FP8−FP16 gap at the final checkpoint.

In [5]:
# Checkpoint-evolution curves (ours only; needs >1 checkpoint to be interesting).
ours_raw = raw_df[(raw_df.series != "authors") & (raw_df.step >= 0)]
for short, spec in MODELS.items():
    for et, metric in [("core", "CE_loss_score"), ("sparse_probing", "probe_top1_acc")]:
        sub = ours_raw[(ours_raw.model == short)
                       & (ours_raw.eval_type == et) & (ours_raw.metric == metric)]
        if sub.empty or sub["step"].nunique() < 2:
            continue
        fig = go.Figure()
        for (series, point), g in sub.groupby(["series", "point"]):
            g = g.sort_values("step")
            fig.add_trace(go.Scatter(
                x=g["step"], y=g["value"], mode="lines+markers",
                name=f"{series} {point}",
                line=dict(dash="dash" if "FP8" in series else "solid")))
        fig.update_layout(title=f"{spec['full']} — {metric} vs training step",
                          xaxis_title="training step", yaxis_title=metric,
                          width=860, height=480)
        fig.show()

In [6]:
# FP8 vs FP16 gap at the final checkpoint, per model x k.
metric_cols = [c for c in ["L0", "CE_loss_score", "explained_var", "KL_div_score",
                           "frac_alive", "probe_top1_acc", "probe_top5_acc"]
               if c in wide.columns]
ours_wide = wide[wide["series"].isin(["FP8 (ours)", "FP16 (ours)"])]
gap_rows = []
for (model, point), g in ours_wide.groupby(["model", "point"]):
    fp8 = g[g.series == "FP8 (ours)"]
    fp16 = g[g.series == "FP16 (ours)"]
    if fp8.empty or fp16.empty:
        continue
    row = {"model": model, "k": point}
    for c in metric_cols:
        v8, v16 = fp8[c].iloc[0], fp16[c].iloc[0]
        row[f"{c}_fp16"] = v16
        row[f"{c}_fp8"] = v8
        row[f"{c}_diff"] = (v8 - v16) if pd.notna(v8) and pd.notna(v16) else None
    gap_rows.append(row)

gap = pd.DataFrame(gap_rows)
if not gap.empty:
    gap["k_int"] = gap["k"].str.lstrip("k").astype(int)
    gap = gap.sort_values(["model", "k_int"]).drop(columns="k_int").reset_index(drop=True)
gap

""
